In [ ]:
import requests
from langchain_community.utilities import SerpAPIWrapper, GoogleSerperAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from pydantic import BaseModel
from langchain.tools import tool
from datetime import datetime,date
from typing import Union
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START,END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
os.environ["SERPER_API_KEY"] = os.getenv("SERPER_API_KEY")
os.environ["OPENWEATHER_API_KEY"] = os.getenv("OPENWEATHER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["SERP_API_KEY"] = os.getenv("SERP_API_KEY")
os.environ["EXCHANGE_RATE_API"] = os.getenv("EXCHANGE_RATE_API")


In [ ]:
os.environ["OPENAI_API_KEY"]

## Tools

In [ ]:
## 1. Weather tool

@tool
def get_current_weather(target_location :str):
    """
    Retrives the current weather conditions for a specified location  using Serper API.

    This tool takes a location as input and returns the current weather conditions, including temperature, humidity, and weather description.
      It is useful for travelers who want to know the weather conditions at their destination before they arrive.

    Parameters:
    - target_location (str): The location for which to retrieve the current weather conditions.

    Returns:
    - str: A string describing the current weather conditions at the specified location.
    """

    google_search = SerpAPIWrapper(serpapi_api_key=os.getenv("SERP_API_KEY"))
    search_query = f"what is the wather in {target_location}?"

    current_weather = google_search.run(search_query)

    return current_weather



@tool
def get_weather_forecast(target_location :str, time: Union[str, date, datetime]):
    """
    Retrieves the weather forecast for a specified location using the SerpAPI .

    This tool takes a location as input and returns the weather forecast for that location, including temperature, humidity,rain and weather description for the upcoming days.
      It is useful for travelers who want to plan their activities based on the expected weather conditions at their destination.

    Parameters:
    - target_location (str): The location for which to retrieve the weather forecast.
    - time (Union[str, date, datetime]): The time for which to retrieve the weather forecast. This can be a string (e.g., "tomorrow", "next week"), a date object, or a datetime object.

    Returns:
    - str: A string describing the weather forecast for the specified location.
    """

    google_search = SerpAPIWrapper(serpapi_api_key=os.getenv("SERP_API_KEY"))
    search_query = f"what is the wather forecast in {target_location} on {time}?"

    predicted_weather = google_search.run(search_query)

    return predicted_weather






In [ ]:
#get_current_weather.invoke("Paris")
#get_weather_forecast.invoke({"target_location": "Chennai","time": "2026-04-23"})

In [ ]:
##  Search Hotel tool


@tool
def search_hotels(destination: str,budget:int)-> str:
    """
    Searches for hotels in a specified destination within a given budget using the GoogleSerprer API.

    This tool takes a destination and a budget as input and returns a list of hotels that match the criteria, including their names, prices, and ratings.
      It is useful for travelers who want to find accommodation options that fit their preferences and budget.

    Parameters:
    - destination (str): The location where the traveler wants to find hotels.
    - budget (int): The maximum price the traveler is willing to pay for a hotel.

    Returns:
    - str: A string describing the hotels that match the search criteria, including their names, prices, and ratings.
    """

    search = GoogleSerperAPIWrapper()
    search_query = f"Find hotels in {destination} with a budget of ${budget} ."

    search_hotels = search.run(search_query)

    return search_hotels


search_hotels.invoke({"destination": "Paris","budget": 100})

In [ ]:
## HOTEL ESTIMATION

@tool
def estimate_hotel_cost(price_per_night: float, total_days:int) -> float:
    """
    Estimates the total cost of a hotel stay based on the destination, number of nights, and budget .

    This tool takes a destination, number of nights and returns an estimated total cost for the hotel stay. 

    Parameters:
    - destination (str): The location where the traveler plans to stay.
    - total_days (int): The number of days the traveler intends to stay at the hotel.

    Returns:
    - flaot: A numeric describing the estimated total cost for the hotel stay based on the provided criteria.
    """
    try:
        total_cost = round(price_per_night * total_days, 2)
        return total_cost
    except Exception as e:
        print(f"Error occurred while estimating hotel cost: {e}")
        return str(e)
    
estimate_hotel_cost.invoke({"price_per_night": 1000, "total_days": 5})

In [ ]:
## TOURIST ATTRACTION SEARCH
@tool
def search_tourist_attractions(destination: str) -> str:
    """
    Searches for tourist attractions in a specified destination using the Tavily Search API.

    This tool takes a destination as input and returns a list of popular tourist attractions in that location, including their names, descriptions, and ratings.
      It is useful for travelers who want to explore the key attractions and points of interest at their destination.

    Parameters:
    - destination (str): The location where the traveler wants to find tourist attractions.

    Returns:
    - str: A string describing the popular tourist attractions in the specified destination, including their names, descriptions, and ratings.
    """

    search = TavilySearchResults(k=5)
    search_query = f"Find popular tourist attractions in {destination}."

    attractions = search.invoke(search_query)

    return "\n".join([r['content'] for r in attractions])


search_tourist_attractions("Chennai")

In [ ]:
## Restaurant Search
@tool
def search_restaurants(destination: str) -> str:
    """
    Searches for restaurants in a specified destination  using the Tavily Search API.

    Parameters:
    - destination (str): The location where the traveler wants to find restaurants.

    Returns:
    - str: A string describing the restaurants that match the search criteria, including their names, locations, and ratings.
    """

    search = TavilySearchResults(k=5)
    search_query = f"Find restaurants to try mandatorily in {destination}."

    restaurants = search.invoke(search_query)

    return "\n".join([r['content'] for r in restaurants])

search_restaurants("Chennai")

In [ ]:
@tool
def search_activities(destination: str) -> str:
    """
    Searches for activities in a specified destination using the Tavily Search API.

    Parameters:
    - destination (str): The location where the traveler wants to find activities.

    Returns:
    - str: A string describing the activities that match the search criteria, including their names, locations, and ratings.
    """

    search = TavilySearchResults(k=5)
    search_query = f"Find activities to do mandatorily in {destination}."

    activities = search.invoke(search_query)

    return "\n".join([r['content'] for r in activities])


@tool
def search_transportation(destination: str) -> str:
    """
    Searches for transportation options in a specified destination using the Tavily Search API.

    Parameters:
    - destination (str): The location where the traveler wants to find transportation options.

    Returns:
    - str: A string describing the transportation like bus, metro, taxi, etc that match the search criteria.
    """

    search = TavilySearchResults(k=5)
    search_query = f"Find transportation options to use mandatorily in {destination}."

    transportation_options = search.invoke(search_query)

    return "\n".join([r['content'] for r in transportation_options])

In [ ]:
## EXCHANGE RATE TOOL
@tool
def get_exchange_rate(source_currency: str, target_currency: str) -> float:
    """
    Retrieves the exchange rate between two specified currencies using the Exchange Rate API.

    This tool takes a source currency and a target currency as input and returns the current exchange rate between them.
      It is useful for travelers who want to know the value of their home currency in terms of the local currency at their destination.

    Parameters:
    - source_currency (str): The currency code of the source currency (e.g., "USD" for US Dollar).
    - target_currency (str): The currency code of the target currency (e.g., "EUR" for Euro).

    Returns:
    - float: The exchange rate between the source and target currencies.
    """

    api_key = os.getenv("EXCHANGE_RATE_API")
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{source_currency}/{target_currency}"

    try:
        response = requests.get(url)
        data = response.json()

        if data['result'] == 'success':
            exchange_rate = data['conversion_rate']
            return f"The current exchange rate from {source_currency} to {target_currency} is: {exchange_rate}"
        else:
            return f"Error retrieving exchange rate: {data['error-type']}"

    except Exception as e:
        return f"An error occurred while fetching exchange rate: {e}"
    

@tool
def convert_currency(amount: float, conversion_rate: float) -> float:
    """
    Converts a specified amount from a source currency to a target currency using the Exchange Rate API.

    This tool takes an amount, a source currency, and a target currency as input and returns the converted amount in the target currency.
      It is useful for travelers who want to know how much their money is worth in the local currency at their destination.

    Parameters:
    - amount (float): The amount of money to convert.
=   - conversion_rate (float): The exchange rate between the source and target currencies.

    Returns:
    - float: The converted amount in the target currency.
    """

    return round(amount * conversion_rate, 2)

In [ ]:
## ARITHMETIC CALCULATOR

@tool
def add(a: float, b: float) -> float:
    """
    Adds two numbers together.

    Parameters:
    - a (float): The first number to add.
    - b (float): The second number to add.

    Returns:
    - float: The sum of the two numbers.
    """
    return round(a + b, 2)


@tool
def subtract(a: float, b: float) -> float:
    """
    Subtracts one number from another.

    Parameters:
    - a (float): The number to be subtracted from.
    - b (float): The number to subtract.

    Returns:
    - float: The difference between the two numbers.
    """
    return round(a - b, 2)

@tool
def multiply(a: float, b: float) -> float:
    """
    Multiplies two numbers together.

    Parameters:
    - a (float): The first number to multiply.
    - b (float): The second number to multiply.

    Returns:
    - float: The product of the two numbers.
    """
    return round(a * b, 2)

@tool
def divide(a: float, b: float) -> float:
    """
    Divides one number by another.

    Parameters:
    - a (float): The number to be divided.
    - b (float): The number to divide by.

    Returns:
    - float: The quotient of the two numbers.
    """
    if b == 0:
        return "Error: Division by zero is not allowed."
    return round(a / b, 2)

In [ ]:
@tool
def calculate_total_cost(hotel_cost: float, activity_cost: float, restaurant_cost: float, transportation_cost: float) -> float:
    """
    Calculates the total cost of a trip based on hotel, activity, restaurant, and transportation costs.

    Parameters:
    - hotel_cost (float): The total cost of the hotel stay.
    - activity_cost (float): The total cost of activities planned for the trip.
    - restaurant_cost (float): The total cost of meals at restaurants during the trip.
    - transportation_cost (float): The total cost of transportation during the trip.

    Returns:
    - float: The total estimated cost of the trip.
    """
    return round(hotel_cost + activity_cost + restaurant_cost + transportation_cost, 2)


@tool
def calculate_daily_budget(total_cost: float, total_days: int) -> float:
    """
    Calculates the daily budget for a trip based on the total estimated cost and the number of days.

    Parameters:
    - total_cost (float): The total estimated cost of the trip.
    - total_days (int): The number of days the trip will last.

    Returns:
    - float: The daily budget for the trip.
    """
    if total_days == 0:
        return "Error: Total days cannot be zero."
    return round(total_cost / total_days, 2)

In [ ]:
SYSTEM_PROMPT = """
You are a AI travel agent & planner assistant that helps users plan their trips by providing information on weather, hotels, tourist attractions, restaurants, transportation options, and currency exchange rates. 
You can also perform basic arithmetic calculations to help users estimate costs and budgets for their trips using real-time data and tools.

You must:
1. Understand the user's travel preferences and requirements, such as destination, travel dates, budget, and interests.
2. Decide which tools to call and in what order based on the user's input and the information needed to provide a comprehensive travel plan.
3. Use tools to fetch:
    - Real time weather (current or forecast )
    - Local attractions, restaurants, activities, transportation
    - Hotel options and estimate hotel costs
    - Add or multiply or subtract or divide to calcualate total cost and budget
    - Convert the total cost to user's currency using real time exchange rate
    - Generate day to day itenary
    - Generate final summary of the full trip plan.


Instructions:
- select one tool at a time base on user input
- Once all data is generated provide a full day by day itenary
- Summarize the travel paln includign location, dates, weather, top places, cost and final recommendation.


End Goal:
Return a complete itenary plan including:
- Weather conditions
- Recommendation attraction, activities and restaurants
- Hotel options and cost
- Daily and total budget and currency conversion
- Day wise iternary
- Natual language summary

Be informative , rely on tools for real time and factual data.
            
    """

In [ ]:
tools = [
    get_current_weather,
    get_weather_forecast,
    search_hotels,
    estimate_hotel_cost,
    search_tourist_attractions,
    search_restaurants,
    search_activities,
    search_transportation,
    get_exchange_rate,
    convert_currency,
    add,
    multiply,
    subtract,
    divide,
    calculate_total_cost,
    calculate_daily_budget
]

In [ ]:
llm = ChatOpenAI(model="gpt-4o",temperature=0.3)

In [ ]:
llm_with_bind_tools = llm.bind_tools(tools)

In [ ]:
llm_with_bind_tools.invoke("Hi! i want a 5 day trip to Dubai next month. I'd like to know what the weather will be like, what places i can visit, and how much thte total trip cost. I want the trip cost in INR. I prefer local food and public transport.")

In [ ]:
def call_model(state: MessagesState):
    question = state["messages"]
    question_with_system_prompt = [SYSTEM_PROMPT] + question

    response    = llm_with_bind_tools.invoke(question_with_system_prompt)
    return{"messages":response}

In [ ]:
## Graph Nodes

builder = StateGraph(MessagesState)

builder.add_node("ai_trip_planner",call_model)
builder.add_node("tools",ToolNode(tools))


builder.add_edge(START,"ai_trip_planner")
builder.add_conditional_edges("ai_trip_planner",tools_condition)
builder.add_edge("tools","ai_trip_planner")

travel_agent = builder.compile()

In [ ]:
from IPython.display import display,Image

display(Image(travel_agent.get_graph().draw_mermaid_png()))

In [ ]:
input = "Hi! i want a 5 day trip to Dubai next month. I'd like to know what the weather will be like, what places i can visit, and how much thte total trip cost. I want the trip cost in INR. I prefer local food and public transport."

In [ ]:
output = travel_agent.invoke({"messages":input})

for o in output["messages"]:    
    o.pretty_print()